# Funness model features

In [ ]:
import os
import json
import random

import numpy as np
import pandas as pd

import analysis_utils
from analysis_utils import process_model_runs
from models.utils import compute_emd, get_outcome_dist
from constants import THINK_HUMAN_DATA, THINK_STIMULI_PTH

MODEL_DIR = "../model-data/think-exp"
INTERMODEL_DIR = "../model-data/intermodel"
LLM_DIR = f"{MODEL_DIR}/llms"
FEATURES_OUT_DIR = "../model-data"
HUMAN_PROCESSED_PTH = "final_processed_res/think/human_processed.json"

np.random.seed(7)
random.seed(7)

In [ ]:
_, idx2game, game2idx, _ = analysis_utils.process_game_stimuli(THINK_STIMULI_PTH)
human_df = pd.read_csv(THINK_HUMAN_DATA)
ordered_games = sorted(set(human_df.game_id))

with open(HUMAN_PROCESSED_PTH) as f:
    human_fun = json.load(f)["human_fun"]

## Load model rollouts (main + challenge-arena variants)

Challenge variants are 1v1 against uni_random — `_ch1` = model is P1,
`_ch2` = model is P2 — used to score "challenge" against a baseline opponent.

Note: we refer to "challenge" as rewardingness to think in the main paper

In [ ]:
model2pth = {
    # main rollouts
    "ours_full":       f"{MODEL_DIR}/full_or_partial/full.txt",
    "expert":          f"{MODEL_DIR}/2025-07-04_heuristic_search_eg.txt",
    "expert_mcts":     f"{MODEL_DIR}/mcts_results_merged.txt",
    "random_terminal": f"{MODEL_DIR}/main_rand_untilend_results_400.txt",
    "offense":         f"{MODEL_DIR}/ablations/heuristics/{{'center_weight': 1.0, 'block_weight': 0.0, 'connect_weight': 1.0}}.txt",
    "defense":         f"{MODEL_DIR}/ablations/heuristics/{{'center_weight': 1.0, 'block_weight': 1.0, 'connect_weight': 0.0}}.txt",

    # challenge arenas (model vs uni_random)
    "ours_challenge1":    f"{MODEL_DIR}/challenge1_results_200_20250111-2304.txt",
    "ours_challenge2":    f"{MODEL_DIR}/challenge2_results_200_20250111-2304.txt",
    "expert_challenge1":  f"{INTERMODEL_DIR}/2025-07-04_{{'agent': 'heuristic_search_eg', 'other_agent': 'uni_random', 'p1': {{}}, 'p2': {{}}}}.txt",
    "expert_challenge2":  f"{INTERMODEL_DIR}/2025-07-04_{{'agent': 'uni_random', 'other_agent': 'heuristic_search_eg', 'p1': {{}}, 'p2': {{}}}}.txt",
    "random_challenge1":  f"{MODEL_DIR}/uni_random.txt",
    "random_challenge2":  f"{MODEL_DIR}/uni_random.txt",
    "offense_challenge1": f"{INTERMODEL_DIR}/{{'agent': 'heuristics', 'other_agent': 'uni_random', 'p1': {{'block_weight': 0}}, 'p2': {{}}}}.txt",
    "offense_challenge2": f"{INTERMODEL_DIR}/{{'agent': 'uni_random', 'other_agent': 'heuristics', 'p1': {{}}, 'p2': {{'block_weight': 0}}}}.txt",
    "defense_challenge1": f"{INTERMODEL_DIR}/{{'agent': 'heuristics', 'other_agent': 'uni_random', 'p1': {{'connect_weight': 0}}, 'p2': {{}}}}.txt",
    "defense_challenge2": f"{INTERMODEL_DIR}/{{'agent': 'uni_random', 'other_agent': 'heuristics', 'p1': {{}}, 'p2': {{'connect_weight': 0}}}}.txt",
    
    "gpt_cot_fun": f"{LLM_DIR}/gpt_responses_fun/agg_how_fun_True_20240130-2253.json",
}

model2res = {}
for model, pth in model2pth.items():
    res = process_model_runs(model, pth, idx2game)
    if res is None:
        raise FileNotFoundError(f"Failed to load {model} from {pth}")
    model2res[model] = res

## Feature builders

In [ ]:

# Agg participants and sims 20x6 -- 120 outcomes per game for computation
K_SUBSAMPLE = 20 * 6 

# `local` is the same as Intuitive Gamer
model_configs = [
    # (csv_name, main_model_key, ch1_key, ch2_key)
    ("local",   "ours_full",       "ours_challenge1",    "ours_challenge2"),
    ("expert",  "expert",          "expert_challenge1",  "expert_challenge2"),
    ("random",  "random_terminal", "random_challenge1",  "random_challenge2"),
    ("offense", "offense",         "offense_challenge1", "offense_challenge2"),
    ("defense", "defense",         "defense_challenge1", "defense_challenge2"),
]

def neg_emd(scores):
    return -compute_emd(get_outcome_dist(scores))

def challenge(ch1_scores, ch2_scores):
    s1 = list(np.random.choice(ch1_scores, K_SUBSAMPLE, replace=True))
    s2 = list(np.random.choice(ch2_scores, K_SUBSAMPLE, replace=True))
    ch1_pwin = s1.count(1) / len(s1)
    ch2_pwin = s2.count(1) / len(s2)
    ch2_pdraw = s2.count(0) / len(s2)
    return (ch1_pwin + (1 - (ch2_pdraw + ch2_pwin))) / 2

## Build + save features per model

In [ ]:
gpt_fun = model2res["gpt_cot_fun"]

for csv_name, main_key, ch1_key, ch2_key in model_configs:
    np.random.seed(7)
    main = model2res[main_key]
    ch1 = model2res[ch1_key]
    ch2 = model2res[ch2_key]

    rows = []
    skipped = []
    for game in ordered_games:
        if game not in main or game not in ch1 or game not in ch2:
            skipped.append(game)
            continue
        if not human_fun.get(game):
            skipped.append(game)
            continue
        scores = main[game]["game_scores"]
        ch_scores = []
        ch_scores.append(challenge(ch1[game]["game_scores"], ch2[game]["game_scores"]))
        rows.append({
            "emd":       neg_emd(scores),
            "challenge": float(np.mean(ch_scores)),
            "len":       float(np.mean(main[game]["game_lengths"])),
            "game_idx":  game2idx[game] - 1,
            "game_id":   game,
            "llm":       gpt_fun[game],
            "humans":    human_fun[game],
        })

    df = (pd.DataFrame(rows)
            .sort_values(by="game_idx")
            .reset_index())
    out_pth = f"{FEATURES_OUT_DIR}/{csv_name}_model_readout_fun_features.csv"
    df.to_csv(out_pth)
    print(f"{csv_name:>8s}: wrote {len(df)} rows -> {out_pth}"
          + (f" (skipped {len(skipped)})" if skipped else ""))